# Unified Image to Anthropometry Pipeline

This notebook starts from the exposed faces of the relocated wrapper and then follows one run through the staged contract.

Primary command-line face:

```bash
python -m unified --input IMAGE_OR_OBJ_OR_DIR --image-method auto --anthro-method auto --units auto
```

Stage command faces:

```bash
python -m unified img2obj --input path/to/person.png --method auto
python -m unified obj2anthro --input path/to/person.obj --method auto --units auto
python -m unified.img2obj --input path/to/person.png --method auto
python -m unified.obj2anthro --input path/to/person.obj --method auto --units auto
```

Python API face:

```python
from unified.pipeline import classify_input, run_pipeline
```

All faces link through the same staging contract: classify inputs, create concrete OBJ handoffs, run anthropometry branches, and write `runs/<run_id>/manifest.json`.

This notebook was verified with the Python executable printed in the next cell. Use that same executable/kernel when reproducing the run.

In [1]:
import sys
print(sys.executable)

C:\Users\Clint\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    if (candidate / "unified").is_dir():
        repo = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the Pennington-MATLAB-Python repository.")
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from unified.pipeline import classify_input, allocate_run_root, run_pipeline

demo_root = repo / "runs" / "notebook_demo"
demo_root.mkdir(parents=True, exist_ok=True)

image_input = demo_root / "person.png"
obj_input = demo_root / "direct_subject.obj"
mixed_dir = demo_root / "mixed_inputs"
mixed_dir.mkdir(exist_ok=True)

image_input.write_bytes(b"notebook image placeholder")
obj_input.write_text("# notebook direct obj\nv 0 0 0\nv 1 0 0\nv 0 1 0\nf 1 2 3\n", encoding="utf-8")
(mixed_dir / "frame.png").write_bytes(b"mixed image placeholder")
(mixed_dir / "scan.obj").write_text(obj_input.read_text(encoding="utf-8"), encoding="utf-8")

print("demo root:", demo_root)

demo root: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo


## Exposed Faces and Links

The wrapper deliberately exposes a small surface area. The top-level CLI is the normal user entry point; stage CLIs are useful for debugging one stage; module CLIs preserve direct stage invocation; Python APIs support notebooks and tests without bypassing wrapper logic.

In [3]:
import pandas as pd

faces = pd.DataFrame([
    {
        "face": "top-level CLI",
        "entry": "python -m unified --input ...",
        "links_to": "unified.pipeline.run_pipeline",
        "artifact": "runs/<run_id>/manifest.json",
    },
    {
        "face": "img2obj stage CLI",
        "entry": "python -m unified img2obj --input ...",
        "links_to": "unified.img2obj.run",
        "artifact": "runs/<run_id>/img2obj/ + OBJ handoffs",
    },
    {
        "face": "obj2anthro stage CLI",
        "entry": "python -m unified obj2anthro --input ...",
        "links_to": "unified.obj2anthro.run_pipeline",
        "artifact": "runs/<run_id>/obj2anthro/.../results.csv",
    },
    {
        "face": "module stage CLIs",
        "entry": "python -m unified.img2obj / python -m unified.obj2anthro",
        "links_to": "same stage wrappers",
        "artifact": "same stage outputs",
    },
    {
        "face": "Python API",
        "entry": "classify_input(), run_pipeline()",
        "links_to": "same pipeline functions used by CLI",
        "artifact": "returned manifest + manifest.json",
    },
])
faces

,face,entry,links_to,artifact
0,top-level CLI,python -m unified --input ...,unified.pipeline.run_pipeline,runs/<run_id>/manifest.json
1,img2obj stage CLI,python -m unified img2obj --input ...,unified.img2obj.run,runs/<run_id>/img2obj/ + OBJ handoffs
2,obj2anthro stage CLI,python -m unified obj2anthro --input ...,unified.obj2anthro.run_pipeline,runs/<run_id>/obj2anthro/.../results.csv
3,module stage CLIs,python -m unified.img2obj / python -m unified....,same stage wrappers,same stage outputs
4,Python API,"classify_input(), run_pipeline()",same pipeline functions used by CLI,returned manifest + manifest.json


## Contract Diagram

```text
IMAGE input
  -> classify_input()
  -> unified.img2obj.run()
  -> obj_handoffs[*].obj_path
  -> unified.obj2anthro.run_pipeline()
  -> obj2anthro/<source_method>/<subject_id>/<anthro_method>/results.csv
  -> manifest.json

OBJ input
  -> classify_input()
  -> direct OBJ handoff
  -> unified.obj2anthro.run_pipeline()
  -> obj2anthro/direct/<subject_id>/<anthro_method>/results.csv
  -> manifest.json

MIXED directory
  -> images follow the image path
  -> direct OBJs follow the direct path
  -> all selected anthropometry methods run for every OBJ handoff
```

## Help Faces

These checks are lightweight and show that the public command surfaces are present before any backend work runs.

In [4]:
import subprocess
import sys

for args in [
    [sys.executable, "-m", "unified", "--help"],
    [sys.executable, "-m", "unified", "img2obj", "--help"],
    [sys.executable, "-m", "unified", "obj2anthro", "--help"],
]:
    proc = subprocess.run(args, cwd=repo, text=True, capture_output=True, check=True)
    print(" ".join(args[1:]))
    print(proc.stdout.splitlines()[0])

-m unified --help
usage: unified [-h] --input INPUT [--image-method IMAGE_METHOD]
-m unified img2obj --help
usage: unified img2obj [-h] --input INPUT [--method METHOD] [--out OUT]


-m unified obj2anthro --help
usage: unified obj2anthro [-h] --input INPUT


## API Faces

The signatures below are the notebook/test entry points. They are thin wrappers around the same CLI contract rather than a second pipeline.

In [5]:
import inspect
import unified.img2obj as img2obj
import unified.obj2anthro as obj2anthro

print("classify_input", inspect.signature(classify_input))
print("run_pipeline", inspect.signature(run_pipeline))
print("img2obj.run", inspect.signature(img2obj.run))
print("obj2anthro.run_pipeline", inspect.signature(obj2anthro.run_pipeline))

classify_input (input_path: 'str | Path') -> 'dict[str, object]'
run_pipeline (input_path: 'str | Path', image_method: 'str' = 'auto', anthro_method: 'str' = 'auto', units: 'str' = 'auto', out: 'str | Path | None' = None) -> 'dict[str, object]'
img2obj.run (input_path: 'str | Path', method: 'str' = 'auto', out: 'str | Path | None' = None, native_args: 'list[str] | None' = None) -> 'dict[str, object]'
obj2anthro.run_pipeline (input_path: 'str | Path' = WindowsPath('C:/Users/Clint/OneDrive/Desktop/py2mat/Pennington-MATLAB-Python/unified/obj2anthro/backends/segmentation/model_files/OBJ'), backend: 'str' = 'all', recursive: 'bool' = True, units: 'str' = 'cm', output_dir: 'str | Path | None' = WindowsPath('C:/Users/Clint/OneDrive/Desktop/py2mat/Pennington-MATLAB-Python/unified/obj2anthro/results'), run_id: 'str | None' = None, n_slices: 'int' = 200, save_images: 'bool' = True, save_aligned_obj: 'bool' = True, height_scale_to_cm: 'float | None' = None, show: 'bool' = False) -> 'pd.DataFrame'

## Input Classification

Images enter `img2obj`, OBJ files go straight to `obj2anthro`, and mixed directories preserve both paths.

In [6]:
for label, path in [("image", image_input), ("obj", obj_input), ("mixed", mixed_dir)]:
    plan = classify_input(path)
    print(f"\n{label}: {path}")
    print(json.dumps({
        "images": plan["images"],
        "objs": plan["objs"],
        "unsupported": plan["unsupported"],
        "subject_groups": plan["subject_groups"],
    }, indent=2))


image: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\person.png
{
  "images": [
    "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\person.png"
  ],
  "objs": [],
  "unsupported": [],
  "subject_groups": {}
}

obj: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\direct_subject.obj
{
  "images": [],
  "objs": [
    "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\direct_subject.obj"
  ],
  "unsupported": [],
  "subject_groups": {}
}

mixed: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\mixed_inputs
{
  "images": [
    "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\mixed_inputs\\frame.png"
  ],
  "objs": [
    "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\mixed_inputs\\scan.obj"
  ],
  "unsupported": 

## Run Directory Allocation

By default the wrapper writes under `runs/<run_id>/`. This notebook uses explicit run roots under `runs/notebook_demo/` so repeated execution is easy to inspect.

In [7]:
allocated = allocate_run_root(demo_root / "allocated_run")
print(allocated)

C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\allocated_run


## Lightweight Stage Backends

The monkeypatches below emulate a successful image-to-OBJ stage and a successful OBJ-to-anthropometry stage. They still enter through `run_pipeline()`, which builds the same root manifest and branch layout used by `python -m unified`.

In [8]:
import unified.img2obj as img2obj
import unified.obj2anthro as obj2anthro

_original_img2obj_run = img2obj.run
_original_obj2anthro_run = obj2anthro.run_pipeline

def fake_img2obj_run(input_path, method="auto", out=None):
    out_dir = Path(out)
    out_dir.mkdir(parents=True, exist_ok=True)
    native_manifest = out_dir / "manifest.json"
    handoff_obj = out_dir / "notebook_subject.obj"
    handoff_obj.write_text("# fake image-derived obj\nv 0 0 0\nv 1 0 0\nv 0 1 0\nf 1 2 3\n", encoding="utf-8")
    native_manifest.write_text(json.dumps({"inputs": [str(input_path)], "backend": method}, indent=2), encoding="utf-8")
    return {
        "status": "success",
        "native_output_dir": str(out_dir),
        "native_manifest_path": str(native_manifest),
        "obj_handoffs": [{
            "subject_id": "notebook_subject",
            "method": "fake_image",
            "obj_path": str(handoff_obj),
            "native_output_dir": str(out_dir),
            "native_manifest_path": str(native_manifest),
            "source_images": [str(input_path)],
            "selected_instance": "notebook_subject",
        }],
        "warnings": [],
        "errors": [],
    }

def fake_obj2anthro_run(input_path, backend="auto", recursive=True, units="auto", output_dir=None, run_id=None, **kwargs):
    output_dir = Path(output_dir)
    raw_dir = output_dir / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / "results.csv"
    row = {"subject_id": Path(input_path).stem, "backend": backend, "units": units, "status": "success"}
    df = pd.DataFrame([row])
    df.to_csv(csv_path, index=False)
    (raw_dir / "backend_manifest.json").write_text(json.dumps({"input": str(input_path), "backend": backend}, indent=2), encoding="utf-8")
    df.attrs["output_csv"] = str(csv_path)
    df.attrs["raw_output_dir"] = str(raw_dir)
    return df

img2obj.run = fake_img2obj_run
obj2anthro.run_pipeline = fake_obj2anthro_run

## Image Input: Handoff to Anthropometry

An image run must produce concrete OBJ handoffs before anthropometry branches run.

In [9]:
image_run = run_pipeline(
    image_input,
    image_method="auto",
    anthro_method="slice",
    units="auto",
    out=demo_root / "image_run",
)
image_manifest_path = Path(image_run["manifest_path"])
print("manifest:", image_manifest_path)
print("status:", image_run["status"])
print(json.dumps(image_run["obj_handoffs"], indent=2))

manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\image_run\manifest.json
status: success
[
  {
    "subject_id": "notebook_subject",
    "source_method": "fake_image",
    "obj_path": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj\\notebook_subject.obj",
    "native_output_dir": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj",
    "native_manifest_path": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj\\manifest.json",
    "source_images": [
      "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\person.png"
    ],
    "selected_instance": "notebook_subject"
  }
]


## Manifest Shapes

The root manifest records image artifacts, OBJ handoffs, and one branch per selected anthropometry method.

In [10]:
manifest = json.loads(image_manifest_path.read_text(encoding="utf-8"))
print("image artifact roots")
print(json.dumps(manifest["image_artifact_roots"], indent=2))
print("\nfirst OBJ handoff")
print(json.dumps(manifest["obj_handoffs"][0], indent=2))
print("\nfirst obj2anthro branch")
print(json.dumps(manifest["stages"]["obj2anthro"][0], indent=2))

image artifact roots
[
  "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj"
]

first OBJ handoff
{
  "subject_id": "notebook_subject",
  "source_method": "fake_image",
  "obj_path": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj\\notebook_subject.obj",
  "native_output_dir": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj",
  "native_manifest_path": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\img2obj\\manifest.json",
  "source_images": [
    "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\person.png"
  ],
  "selected_instance": "notebook_subject"
}

first obj2anthro branch
{
  "source_obj": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\image_run\\im

## Direct OBJ Input Skips `img2obj`

Direct OBJ handoffs use source method `direct` and write straight to `obj2anthro/direct/<subject>/<method>/`.

In [11]:
direct_run = run_pipeline(
    obj_input,
    anthro_method="slice",
    units="auto",
    out=demo_root / "direct_run",
)
print("manifest:", direct_run["manifest_path"])
print("has img2obj stage:", "img2obj" in direct_run["stages"])
print(json.dumps(direct_run["stages"]["obj2anthro"][0], indent=2))

manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\direct_run\manifest.json
has img2obj stage: False
{
  "source_obj": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\direct_subject.obj",
  "source_method": "direct",
  "subject_id": "direct_subject",
  "anthro_method": "slice",
  "branch_dir": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\direct_run\\obj2anthro\\direct\\direct_subject\\slice",
  "output_csv": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\direct_run\\obj2anthro\\direct\\direct_subject\\slice\\results.csv",
  "raw_output_dir": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\direct_run\\obj2anthro\\direct\\direct_subject\\slice\\raw",
  "raw_artifact_dir": "C:\\Users\\Clint\\OneDrive\\Desktop\\py2mat\\Pennington-MATLAB-Python\\runs\\notebook_demo\\di

## Mixed Directory

Mixed directories send direct OBJs straight to anthropometry and send images through `img2obj` first.

In [12]:
mixed_run = run_pipeline(
    mixed_dir,
    image_method="auto",
    anthro_method="slice",
    units="auto",
    out=demo_root / "mixed_run",
)
print("manifest:", mixed_run["manifest_path"])
print("handoffs:", len(mixed_run["obj_handoffs"]))
for branch in mixed_run["stages"]["obj2anthro"]:
    print(branch["source_method"], branch["subject_id"], branch["output_csv"])

manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\mixed_run\manifest.json
handoffs: 2
direct scan C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\mixed_run\obj2anthro\direct\scan\slice\results.csv
fake_image notebook_subject C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\mixed_run\obj2anthro\fake_image\notebook_subject\slice\results.csv


## Inspect Final Outputs

Downstream automation should read `manifest.json` for final CSV and raw artifact locations rather than guessing paths.

In [13]:
for label, run in [("image", image_run), ("direct", direct_run), ("mixed", mixed_run)]:
    root_manifest = Path(run["manifest_path"])
    loaded = json.loads(root_manifest.read_text(encoding="utf-8"))
    print(f"\n{label} manifest: {root_manifest}")
    for branch in loaded["stages"].get("obj2anthro", []):
        print("  csv:", branch["output_csv"])
        print("  raw:", branch["raw_artifact_dir"])


image manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\image_run\manifest.json
  csv: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\image_run\obj2anthro\fake_image\notebook_subject\slice\results.csv
  raw: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\image_run\obj2anthro\fake_image\notebook_subject\slice\raw

direct manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\direct_run\manifest.json
  csv: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\direct_run\obj2anthro\direct\direct_subject\slice\results.csv
  raw: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\direct_run\obj2anthro\direct\direct_subject\slice\raw

mixed manifest: C:\Users\Clint\OneDrive\Desktop\py2mat\Pennington-MATLAB-Python\runs\notebook_demo\mixed_run\manifest.json
  csv: C:\Users\Clint\On

## Optional Real Backend Run

When local CameraHMR/SMPL/image dependencies are installed, run the CLI below from the repository root with a real image. Leave this cell disabled for lightweight notebook verification.

```bash
python -m unified --input unified/docs/assets/ssp3d_beach_volleyball_frame.png --image-method auto --anthro-method auto --units auto
```

In [14]:
# Restore wrapper functions for any later cells in this kernel.
img2obj.run = _original_img2obj_run
obj2anthro.run_pipeline = _original_obj2anthro_run
print("restored stage wrappers")

restored stage wrappers
